# Introduction

Following the functional parcellation of the VMPFC, this notebook focuses on the functional characterization and network connectivity of the resulting subregions. Focusing primarily on the $K=3$ solution, we employ Neurosynth's meta-analytic database to:
1. **Decode** the functional profiles of individual VMPFC clusters using a predefined set of cognitive terms.
2. **Map** the whole-brain co-activation networks associated with each specific subregion.
3. **Characterize** these distributed co-activation networks to validate the functional distinctness of the VMPFC subdivisions (e.g., Valuation, Social cognition, and Emotion).

In [ ]:
import nibabel as nib
import os
from neurosynth import decode
from neurosynth import Dataset
from neurosynth.analysis.cluster import Clusterable
from neurosynth.analysis.network import coactivation
import numpy as np
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
import pandas as pd


plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'Arial'
warnings.filterwarnings("ignore", category=FutureWarning)

DATA_PATH = Path('../data')
PLOTS_PATH = Path('../plots')
RESULTS_PATH = Path('../results')

DATASET_FILE = DATA_PATH / 'neurosynth_data/dataset.pkl'
SURF_FILE = "/home/guoqiu/guoqiu/Database/HCP_S1200_GroupAvg_v1/S1200.L.midthickness_MSMAll.32k_fs_LR.surf.gii"

# Extracting Individual Cluster Masks

The previous K-Means clustering step generated single NIfTI images containing multiple integer labels (e.g., 1, 2, 3 for the $K=3$ solution). To independently analyze the functional profile and connectivity of each distinct VMPFC subregion, we first iterate through the parcellation results and binarize these multi-class images into separate, single-ROI masks.

In [ ]:
for img_file in (RESULTS_PATH / 'nii').iterdir():
    img = nib.load(img_file)
    img_data = np.round(img.get_fdata()).astype(int)
    max_n = img_data.max()
    for i in range(1, max_n + 1):
        new_img_data = (img_data == i).astype(int)
        new_img = nib.Nifti1Image(new_img_data, affine=img.affine, dtype=np.int8)
        new_img_file = RESULTS_PATH / 'nii_by_label' / f'{img_file.name[:-7]}_label{i}.nii.gz'
        nib.save(new_img, new_img_file)

# Functional Decoding of VMPFC Subregions

To understand the cognitive processes associated with each cluster, we perform quantitative functional decoding. By correlating the spatial map of our $K=3$ subregions with Neurosynth's term-based meta-analytic maps (restricted to a predefined list of 15 highly relevant terms in `top15_term.csv`), we can assign specific functional domains—such as valuation, social cognition, and emotion—to distinct anatomical clusters.

In [ ]:
dataset = Dataset.load(DATASET_FILE)
term_list = pd.read_csv(RESULTS_PATH / 'top15_term.csv')['term'].tolist()
decoder = decode.Decoder(dataset=dataset, features=term_list)


image_list = [str((RESULTS_PATH / f'nii_by_label/K3_label{i}.nii.gz').resolve())
              for i in range(1, 4)]
decode_15_df = decoder.decode(images=image_list)
decode_15_df.columns = ['valuation', 'social', 'emotion']
decode_15_df.to_csv(RESULTS_PATH / 'Neurosynth_decode/top15_terms.csv')
decode_15_df

In [ ]:
# fast glance on result
fig, ax = plt.subplots()

x = np.arange(len(decode_15_df))
bar_width = 0.2
plt.bar(x - bar_width, decode_15_df['social'], width=bar_width, label='social')
plt.bar(x, decode_15_df['emotion'], width=bar_width, label='affect')
plt.bar(x + bar_width, decode_15_df['valuation'], width=bar_width, label='valuation')

plt.xticks(x, decode_15_df.index, rotation=90)
plt.legend()

# Network Decoding

Beyond local voxel decoding, we assess the broader functional networks associated with each subregion. We use the isolated $K=3$ cluster masks as seeds to compute whole-brain co-activation maps, identifying regions that consistently co-activate with each VMPFC subdivision across studies (using a conservative threshold of `0.01`).

Subsequently, we apply the same term-based decoding to these distributed co-activation maps (`pFgA_given_pF=0.50`). This step helps us validate whether distinct VMPFC subregions are embedded within functionally distinct whole-brain circuits.

In [ ]:
image = nib.load(str(RESULTS_PATH / f'nii/K3.nii.gz'))

for label_i in range(1, 4):
    seed = str((RESULTS_PATH / 'nii_by_label' / f'K3_label{label_i}.nii.gz').resolve())
    coactivation(
    dataset=dataset,
    seed=seed,
    threshold=0.01,
    output_dir=RESULTS_PATH / 'coactivation',
    prefix = f'K3_label{label_i}',)

In [ ]:
image_list = [str((RESULTS_PATH / f'coactivation/K3_label{i}_pFgA_given_pF=0.50.nii.gz').resolve())
              for i in range(1, 4)]

decode_15_df = decoder.decode(images=image_list)
decode_15_df.columns = ['valuation', 'social', 'emotion']
decode_15_df.to_csv(RESULTS_PATH / 'Neurosynth_decode/top15_terms_coactivation.csv')
decode_15_df

In [ ]:
# fast glance on result
fig, ax = plt.subplots()

x = np.arange(len(decode_15_df))
bar_width = 0.2
plt.bar(x - bar_width, decode_15_df['social'], width=bar_width, label='social')
plt.bar(x, decode_15_df['emotion'], width=bar_width, label='emotion')
plt.bar(x + bar_width, decode_15_df['valuation'], width=bar_width, label='valuation')

plt.xticks(x, decode_15_df.index, rotation=90)
plt.legend()